## Question 1

In [1]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from collections import deque
from typing import Any, Dict, Iterable, List, Optional, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

## Question 2

In [2]:
class Problem(ABC):

    @abstractmethod
    def initial_state(self) -> Any:
      pass

    @abstractmethod
    def is_goal(self, state: Any) -> bool:
      pass

    @abstractmethod
    def actions(self, state: Any) -> List[Any]:
      pass

    @abstractmethod
    def result(self, state: Any, action: Any) -> Any:
      pass

    @abstractmethod
    def action_cost(self, state: Any, action: Any, next_state: Any) -> float:
      pass



## Question 3

In [3]:
@dataclass
class Node:
    state: Any
    parent: Optional["Node"] = None
    action: Optional[Any] = None
    path_cost: float = 0
    depth: int = 0

    def __post_init__(self):
        if self.parent is not None:
            self.depth = self.parent.depth + 1


@dataclass
class SearchResult:
    algorithm: str
    status: str
    solution: Optional[Node]
    nodes_expanded: int
    max_frontier_size: int
    reached_count: int = 0
    limit: Optional[int] = None
    iterations: Optional[List[Dict[str, Any]]] = None

    @property
    def path(self) -> Optional[List[Any]]:
        if self.solution is None:
            return None
        return reconstruct_path(self.solution)

    @property
    def solution_depth(self) -> Optional[int]:
        if self.solution is None:
            return None
        return self.solution.depth

    @property
    def solution_cost(self) -> Optional[float]:
        if self.solution is None:
            return None
        return self.solution.path_cost

## Question 4

In [4]:
def reconstruct_path(node: Node) -> List[Any]:
    """Returns the list of states from the root node to this node."""
    path = []

    while node is not None:
        path.append(node.state)
        node = node.parent

    path.reverse()
    return path


def reconstruct_actions(node: Node) -> List[Any]:
    """Returns the list of actions from the root node to this node."""
    actions = []

    while node is not None and node.parent is not None:
        actions.append(node.action)
        node = node.parent

    actions.reverse()
    return actions


def state_is_on_path(node: Node, state: Any) -> bool:
    """
    Returns True if state already appears on the path from the root to node.

    This is useful for depth-limited search because DLS often uses path-cycle
    checking instead of a global reached set.
    """
    while node is not None:
        if node.state == state:
            return True
        node = node.parent

    return False


def result_to_row(result: SearchResult) -> Dict[str, Any]:
    """Converts a SearchResult object into a row for a pandas DataFrame."""
    return {
        "Algorithm": result.algorithm,
        "Status": result.status,
        "Limit": result.limit,
        "Solution depth": result.solution_depth,
        "Solution cost": result.solution_cost,
        "Nodes expanded": result.nodes_expanded,
        "Max frontier/stack": result.max_frontier_size,
        "Reached states": result.reached_count,
    }


def show_results(results: List[SearchResult]) -> pd.DataFrame:
    """Display results as a DataFrame."""
    return pd.DataFrame([result_to_row(r) for r in results])

## Question 5

In [5]:
MOVES = {
    "UP": (-1, 0),
    "DOWN": (1, 0),
    "LEFT": (0, -1),
    "RIGHT": (0, 1),
}


class GridProblem(Problem):
    def __init__(
        self,
        grid: List[List[int]],
        start: Tuple[int, int],
        goal: Tuple[int, int],
    ):
        """
        grid:
            2D list where 0 = free cell and 1 = obstacle.

        start, goal:
            Tuples in the form (row, col).
        """
        self.grid = grid
        self.start = start
        self.goal = goal

        self.rows = len(grid)
        self.cols = len(grid[0])

    def initial_state(self) -> Tuple[int, int]:
        return self.start

    def is_goal(self, state: Tuple[int, int]) -> bool:
        # TODO 1:
        return state == self.goal

    def in_bounds(self, state: Tuple[int, int]) -> bool:
        row, col = state
        return 0 <= row < self.rows and 0 <= col < self.cols

    def is_free(self, state: Tuple[int, int]) -> bool:
        row, col = state
        return self.grid[row][col] == 0

    def actions(self, state: Tuple[int, int]) -> List[str]:
        # TODO 2:

        legal_actions = []
        row, col = state

        for action, (row_change, col_change) in MOVES.items():
          next_state = (row + row_change, col + col_change)

          if self.in_bounds(next_state) and self.is_free(next_state):
            legal_actions.append(action)

        return legal_actions

    def result(self, state: Tuple[int, int], action: str) -> Tuple[int, int]:
        # TODO 3:

        row, col = state
        dr, dc = MOVES[action]
        return (row + dr, col + dc)

    def action_cost(
        self,
        state: Tuple[int, int],
        action: str,
        next_state: Tuple[int, int],
    ) -> float:
        # TODO 4:

        return 1.0

## Question 5.1

In [6]:
test_grid = [
    [0, 0, 0],
    [1, 1, 0],
    [0, 0, 0],
]

test_problem = GridProblem(test_grid, start=(0, 0), goal=(2, 2))

assert test_problem.initial_state() == (0, 0)
assert test_problem.is_goal((2, 2)) is True
assert test_problem.is_goal((0, 0)) is False
assert test_problem.actions((0, 0)) == ["RIGHT"]
assert test_problem.result((0, 0), "RIGHT") == (0, 1)
assert test_problem.action_cost((0, 0), "RIGHT", (0, 1)) == 1

print("GridProblem self-check passed.")

GridProblem self-check passed.


## Question 6

In [7]:
sample_grid = [
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [1, 1, 1, 0, 1, 0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    [0, 1, 1, 1, 1, 0, 1, 0, 1, 1],
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 1, 1, 1, 1, 1, 0],
    [0, 0, 1, 0, 0, 0, 0, 0, 1, 0],
    [1, 0, 1, 1, 1, 1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    [0, 1, 1, 1, 1, 0, 0, 0, 1, 0],
]

start = (0, 0)
goal = (9, 9)

problem = GridProblem(sample_grid, start, goal)

## Question 8

In [ ]:
def plot_path(
    grid: List[List[int]],
    start: Optional[Tuple[int, int]] = None,
    goal: Optional[Tuple[int, int]] = None,
    path: Optional[List[Tuple[int, int]]] = None,
    terrain_costs: Optional[List[List[float]]] = None,
    title: str = "Grid Map",
):
    """Visualise a grid and, optionally, a solution path."""
    arr = np.array(grid)
    height, width = arr.shape

    path_set = set(path) if path is not None else set()

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.set_xlim(0, width)
    ax.set_ylim(height, 0)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title)

    for row in range(height):
        for col in range(width):
            state = (row, col)

            if arr[row, col] == 1:
                fill = (0.15, 0.15, 0.15)
            elif start is not None and state == start:
                fill = (0.95, 0.20, 0.20)
            elif goal is not None and state == goal:
                fill = (0.20, 0.70, 0.25)
            elif state in path_set:
                fill = (0.95, 0.90, 0.35)
            else:
                fill = (0.95, 0.95, 0.95)

            rect = patches.Rectangle(
                (col, row),
                1,
                1,
                linewidth=1,
                edgecolor=(0.75, 0.75, 0.75),
                facecolor=fill,
            )
            ax.add_patch(rect)

            if terrain_costs is not None and arr[row, col] == 0:
                ax.text(
                    col + 0.5,
                    row + 0.5,
                    str(terrain_costs[row][col]),
                    ha="center",
                    va="center",
                    fontsize=8,
                )

    plt.show()

class SearchAlgorithm(ABC):
    """Base class for search algorithms."""

    def expand(self, problem: Problem, node: Node) -> Iterable[Node]:
        # TODO 5:

        s = node.state

        for action in problem.actions(s):
          s_prime = problem.result(s, action)
          cost = node.path_cost + problem.action_cost(s, action, s_prime)
          yield Node(state = s_prime, parent = node, action = action, path_cost = cost)

    @abstractmethod
    def search(self, problem: Problem) -> SearchResult:
        pass


## Question 9

class BreadthFirstSearch(SearchAlgorithm):
  def search(self, problem: Problem) -> SearchResult:
    algorithm = "BFS"

    # TODO 6:
    node = Node(problem.initial_state())

    if problem.is_goal(node.state):
      return node

    frontier = deque([node])
    reached = {node.state}

    nodes_expanded = 0
    max_frontier_size = len(frontier)


    while frontier:
      node = frontier.popleft()

      for child in self.expand(problem, node):
        s = child.state

        if problem.is_goal(s):
          return SearchResult(
              algorithm=algorithm,
              status="solution_found",
              solution=child,
              nodes_expanded=nodes_expanded,
              max_frontier_size=max_frontier_size,
              reached_count=len(reached)
          )

        if s not in reached:
          reached.add(s)
          frontier.append(child)
          if len(frontier) > max_frontier_size:
              max_frontier_size = len(frontier)
        nodes_expanded += 1


    return SearchResult(
        algorithm=algorithm,
        status="no_solution",
        solution=None,
        nodes_expanded=nodes_expanded,
        max_frontier_size=max_frontier_size,
        reached_count=len(reached)
    )


## Question 10
class DepthFirstSearch(SearchAlgorithm):
  def search(self, problem: Problem) -> SearchResult:
    algorithm = "DFS"

    node = Node(problem.initial_state())

    if problem.is_goal(node.state):
      return node

    frontier = [node]
    reached = {node.state}

    nodes_expanded = 0
    max_frontier_size = len(frontier)


    while frontier:
      node = frontier.pop()

      for child in self.expand(problem, node):
        s = child.state

        if problem.is_goal(s):
          return SearchResult(
              algorithm=algorithm,
              status="solution_found",
              solution=child,
              nodes_expanded=nodes_expanded,
              max_frontier_size=max_frontier_size,
              reached_count=len(reached)
          )

        if s not in reached:
          reached.add(s)
          frontier.append(child)
          if len(frontier) > max_frontier_size:
              max_frontier_size = len(frontier)
        nodes_expanded += 1

    return SearchResult(
        algorithm=algorithm,
        status="no_solution",
        solution=None,
        nodes_expanded=nodes_expanded,
        max_frontier_size=max_frontier_size,
        reached_count=len(reached)
    )


## Question 11

class DepthLimitedSearch(SearchAlgorithm):
  def search(self, problem: Problem, limit: int = 10) -> SearchResult:
      algorithm = "DLS"

      initial_node = Node(problem.initial_state())

      metrics = {
          "nodes_expanded": 0,
          "max_stack_size": 1,
      }

      solution, status = self._recursive_dls(
          problem=problem,
          node=initial_node,
          limit=limit,
          metrics=metrics,
          current_stack_size=1,
      )

      return SearchResult(
          algorithm=algorithm,
          status=status,
          solution=solution,
          nodes_expanded=metrics["nodes_expanded"],
          max_frontier_size=metrics["max_stack_size"],
          reached_count=0,
          limit=limit,
      )

  def _recursive_dls(
      self,
      problem: Problem,
      node: Node,
      limit: int,
      metrics: Dict[str, int],
      current_stack_size: int,
  ) -> Tuple[Optional[Node], str]:

    if problem.is_goal(node.state):
        return node, "success"


    if node.depth >= limit:
        return None, "cutoff"


    metrics["nodes_expanded"] += 1
    cutoff_occurred = False


    for child in self.expand(problem, node):

        in_path = False
        current_ancestor = node
        while current_ancestor is not None:
            if current_ancestor.state == child.state:
                in_path = True
                break

            current_ancestor = getattr(current_ancestor, 'parent', None)

        if in_path:
            continue


        metrics["max_stack_size"] = max(metrics["max_stack_size"], current_stack_size + 1)


        result_node, status = self._recursive_dls(
            problem=problem,
            node=child,
            limit=limit,
            metrics=metrics,
            current_stack_size=current_stack_size + 1
        )


        if status == "success":
            return result_node, "success"


        if status == "cutoff":
            cutoff_occurred = True

    if cutoff_occurred:
        return None, "cutoff"
    else:
        return None, "failure"


## Question 12
class IterativeDeepeningSearch(SearchAlgorithm):
    def search(self, problem: Problem, max_depth: int = 50) -> SearchResult:
        algorithm = "IDS"

        iteration_log = []
        total_nodes_expanded = 0
        global_max_stack_size = 0

        dls = DepthLimitedSearch()

        for limit in range(max_depth + 1):

            dls_result = dls.search(problem, limit=limit)

            total_nodes_expanded += dls_result.nodes_expanded
            global_max_stack_size = max(global_max_stack_size, dls_result.max_frontier_size)

            iteration_log.append({
                "limit": limit,
                "status": dls_result.status,
                "nodes_expanded": dls_result.nodes_expanded
            })

            if dls_result.status == "success":
                return SearchResult(
                    algorithm=algorithm,
                    status="success",
                    solution=dls_result.solution,
                    nodes_expanded=total_nodes_expanded,
                    max_frontier_size=global_max_stack_size,
                    reached_count=0,
                    limit=limit
                )

            elif dls_result.status == "failure":

                return SearchResult(
                    algorithm=algorithm,
                    status="failure",
                    solution=None,
                    nodes_expanded=total_nodes_expanded,
                    max_frontier_size=global_max_stack_size,
                    reached_count=0,
                    limit=limit
                )

        return SearchResult(
            algorithm=algorithm,
            status="cutoff",
            solution=None,
            nodes_expanded=total_nodes_expanded,
            max_frontier_size=global_max_stack_size,
            reached_count=0,
            limit=max_depth
        )

bfs = BreadthFirstSearch()
dfs = DepthFirstSearch()
dls = DepthLimitedSearch()
ids = IterativeDeepeningSearch()

results = [
    bfs.search(problem),
    dfs.search(problem),
    dls.search(problem, limit=10),
    ids.search(problem, max_depth=30),
]

show_results(results)

bfs_result = results[0]
dfs_result = results[1]

plot_path(
    sample_grid,
    start,
    goal,
    path=bfs_result.path,
    title="BFS Solution Path",
)

plot_path(
    sample_grid,
    start,
    goal,
    path=dfs_result.path,
    title="DFS Solution Path",
)

## Custom Map 1 - many dead ends

custom_grid_1 = [
    [0, 0, 1, 0, 0],
    [0, 1, 1, 0, 0],
    [0, 0, 0, 1, 0],
    [1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 1, 1, 1],
    [0, 0, 0, 0, 0],
    [1, 1, 1, 1, 0],
    [0, 0, 0, 0, 0]
]

custom_start_1 = (0,0)
custom_goal_1 = (0,1)

custom_problem_1 = GridProblem(custom_grid_1, custom_start_1, custom_goal_1)
custom_result_1 = [
    bfs.search(custom_problem_1),
    dfs.search(custom_problem_1),
    dls.search(custom_problem_1, limit=10),
    ids.search(custom_problem_1, max_depth=30),
]

show_results(custom_result_1)

plot_path(
    custom_grid_1,
    custom_start_1,
    custom_goal_1,
    path=custom_result_1[0].path,
    title="BFS Solution Path",
)

plot_path(
    custom_grid_1,
    custom_start_1,
    custom_goal_1,
    path=custom_result_1[3].path,
    title="IDS Solution Path",
)


# Custom Map 2 - Large map, Long Narrow Corridor
custom_grid_2 = [
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1],
    [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0],
    [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0],
    [0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0],
    [0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
    [0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]
]

custom_start_2 = (0, 0)
custom_goal_2 = (14, 14)

custom_problem_2 = GridProblem(custom_grid_2, custom_start_2, custom_goal_2)

custom_results_2 = [
    bfs.search(custom_problem_2),
    dfs.search(custom_problem_2),
    dls.search(custom_problem_2, limit=20),
    ids.search(custom_problem_2, max_depth=60),
]

show_results(custom_results_2)





## Reflection Questions

## 15.1

1. A state shows the current situation of the drone. In this lab, it is shown using the coordinates on the grid.

2. An action refers to a legal move that the drone can make from its current state.

3. The result function returns a new state using a set of rules, given a state and an action.

4. Seperating the problem definition from the search algorithm, helps you to write a search algorithm once and use it to solve varying problems, by passing a different Problem object.

## 15.2

1. A FIFO queue allows the Breadth-first search algorithm to search nodes that are directly a level below it, before checking the levels that 2 steps or more away from the root node.

2. Because it searches the nodes level-by-level, once it finds it's goal, it's unlikely that there would be a shorter path.

3. The reached set serves as a memory for the drone, and it prevents the drone from revisiting the same squares infinitely, because once a square is in the set, the drone ignores that square.

## 15.3

1. A stack uses the algorithm, "last-in-first-out". This allows the algorithm to search the deepest nodes first until it reaches a dead-end, before it checks other options.

2. No, since DFS searches the deepest nodes first, it would ignore the goal if it is right below the root node. Therefore it ignores optimality.

3. DFS uses less memory compared to BFS since it only keeps the memory of the current path it is taking.

4. In a scenario where there are no deep nodes, DFS will end up in a time-consuming search of nodes that do not exist, since it searches nodes at the deepest level first.

## 15.4

1. DFS will fail to find the goal if the depth limit is too small.

2. It refers to the limit to which an algorithm reaches, to search for a goal.

3. DLS contains a limit, which restricts it from checking nodes beyond the limit specified, however DFS works without any limits or restrictions.

4. Since DLS tries to save memory, it does not a global reached set, which helps it keep memory of nodes already visited. Therefore, it continuosly checks the path it is on to ensure that it does not continuously move along the same path until it hits it's limit.

## 15.5

1. It does this in order to search a space layer by layer.

2. This is because, IDS continuosly increases its limit, even though for example, limit 3 fails until it reaches the limit where the goal is, for example 50.

3. Because it is still running the DLS algorithm, it only needs to store a single path in memory at a time.

4. The cost of repeatedly searching from the root does not differ significantly from the cost of searching from the top, when using IDS.

## 15.6

1. In a real world scenario, a location with a lot of obstacles in the air, like trees, buildings, poles, and strong winds would make it more difficult to search.

2. I would select BFS, since it is guaranteed to find the goal in the shortest number of steps, even though the costs are equal.

3. I would select DLS, since it allows for a limit to be set, reducing time inefficiency.

4. Drones fly in 3D space, and not in 2D spaces. Therefore there are many factors which affect the search performance of a drone, such as momentum, wind resistance, which cannot be represented on a grid.